In [4]:
%load_ext autoreload
%autoreload 2

import sys

sys.path.append(f"./../")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
import networkx as nx
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Operator, SparsePauliOp, Pauli
from src.graphs import StaticGraph, DynamicGraph, IntersectingEdgesGraph, MultiEdgeGraph
from src.misc import graph_to_bitstring_edges
import numpy as np
import itertools

In [6]:
T = 200
delta_t = 0.1

def graph_to_bitstring_edges(graph):
    num_nodes = len(graph.nodes)
    num_bits = len(bin(num_nodes - 1)) - 2
    node_to_bitstring = {node: format(node, f'0{num_bits}b') for node in graph.nodes}
    edges_bitstring = {(node_to_bitstring[u], node_to_bitstring[v]) for u, v in graph.edges}
    return edges_bitstring

def unitary_to_pauli(U):
    n = int(np.log2(U.shape[0]))
    dim = 2**n
    pauli_strings = []
    coeffs = []
    for pauli_string in [''.join(p) for p in itertools.product('IXYZ', repeat=n)]:
        P = Pauli(pauli_string)
        P_op = Operator(P).data
        coeff = np.trace(P_op.conj().T @ U) / dim
        if not np.isclose(coeff, 0, atol=1e-10):
            pauli_strings.append(pauli_string)
            coeffs.append(coeff)
    return SparsePauliOp(pauli_strings, coeffs)

def get_dynamic_walk_circuit(edges, T, delta_t):
    G = StaticGraph(edges)
    graph_sequence = [(G, T)]
    dyn_G = DynamicGraph(graph_sequence)
    intersecting_G = IntersectingEdgesGraph(edges)
    graph_sequence = [(graph, delta_t) for graph in intersecting_G.subgraphs]
    dyn_G_approx = DynamicGraph(graph_sequence)
    
    big_qc = QuantumCircuit(G.n_qubits)
    for sq in dyn_G_approx.graph_sequence:
        G = MultiEdgeGraph(sq[0].edges)
        sub_qc = G.get_qc(simplified=True)
        big_qc = big_qc.compose(sub_qc)
    transpiled_qc = transpile(big_qc, basis_gates=['cx', 'u3'], optimization_level=3)
    return transpiled_qc

def circuit_to_pauli_decomposition(circuit):
    op = Operator(circuit)
    unitary = op.data
    return unitary_to_pauli(unitary)

def compact_pauli_string(sparse_pauli_op, delta_t):
    pauli_strings = []
    for pauli_string, coeff in zip(sparse_pauli_op.paulis.to_labels(), sparse_pauli_op.coeffs):
        if np.abs(coeff) > delta_t:  # Only include Pauli strings with coefficients greater than delta_t
            pauli_strings.append(pauli_string)
    
    return pauli_strings


input_file = '../data/graphs/graph8c.g6'
output_file = '../data/pauli_strings.txt'



with open(input_file, 'r') as infile, open(output_file, 'w') as outfile:
    for i, line in enumerate(infile):
        g6_string = line.strip()
        graph = nx.from_graph6_bytes(g6_string.encode('utf-8'))
        bitstring_edges = graph_to_bitstring_edges(graph)
        
        # Get the Pauli string from unitary_to_pauli
        G = StaticGraph(bitstring_edges)
        pauli_op = unitary_to_pauli(G.get_adj_mat())
        pauli_string_unitary = compact_pauli_string(pauli_op, delta_t)
        
        # Get the Pauli string for the circuit from get_dynamic_walk_circuit
        dynamic_circuit = get_dynamic_walk_circuit(bitstring_edges, T, delta_t)
        pauli_op_dynamic = circuit_to_pauli_decomposition(dynamic_circuit)
        pauli_string_dynamic = compact_pauli_string(pauli_op_dynamic, delta_t)
        
        # Write the information to the output file
        outfile.write(f"Graph {i+1}:{bitstring_edges}\n")
        outfile.write(f"Pauli Decompositions: {pauli_string_unitary}\n")
        outfile.write(f"Dynamic Quantum Walk: {pauli_string_dynamic}\n")
        outfile.write("------------------------------------------------------------------------------------------------------------------------------------------------------------\n")

print(f"Analysis completed. Results written to {output_file}")

Analysis completed. Results written to ../data/pauli_strings.txt
